In [1]:
"""
Phase 4: Churn Prediction Models

Goal: Build and compare predictive models against the SQL scorecard
baseline (29x separation between Critical and Low tiers).
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

import xgboost as xgb
import shap

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=FutureWarning)

# Seaborn styling
sns.set_style("whitegrid")
sns.set_palette("husl")

print("Imports complete.")
print(f"pandas {pd.__version__}, scikit-learn (check), xgboost {xgb.__version__}")

Imports complete.
pandas 3.0.3, scikit-learn (check), xgboost 3.2.0


In [2]:
# Load DB credentials from .env (same as ETL script)
load_dotenv('../.env')  # .env is in the project root, one level up

DB_USER = os.getenv("POSTGRES_USER")
DB_PASS = quote_plus(os.getenv("POSTGRES_PASSWORD"))
DB_HOST = os.getenv("POSTGRES_HOST")
DB_PORT = os.getenv("POSTGRES_PORT")
DB_NAME = os.getenv("POSTGRES_DB")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Test the connection
with engine.connect() as conn:
    result = pd.read_sql("SELECT COUNT(*) AS n FROM customers", conn)
    print(f"✓ Connected to PostgreSQL — {result['n'][0]:,} customers in database")

✓ Connected to PostgreSQL — 7,043 customers in database


In [3]:
# Pull the full modeling dataset by joining the 5 normalized tables
modeling_query = """
SELECT
    c.customer_id,

    -- Demographics
    c.gender,
    c.senior_citizen,
    c.has_partner,
    c.has_dependents,

    -- Subscription
    sub.tenure_months,
    sub.contract_type,
    sub.paperless_billing,
    sub.payment_method,

    -- Services
    s.phone_service,
    s.multiple_lines,
    s.internet_service,
    s.online_security,
    s.online_backup,
    s.device_protection,
    s.tech_support,
    s.streaming_tv,
    s.streaming_movies,

    -- Billing
    b.monthly_charges,
    b.total_charges,

    -- Target
    cs.churned
FROM       customers     c
JOIN       subscriptions sub ON c.customer_id = sub.customer_id
JOIN       services      s   ON c.customer_id = s.customer_id
JOIN       billing       b   ON c.customer_id = b.customer_id
JOIN       churn_status  cs  ON c.customer_id = cs.customer_id;
"""

df = pd.read_sql(modeling_query, engine)

print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['churned'].mean():.1%}")
print(f"\nColumn data types:")
print(df.dtypes)

Dataset shape: (7043, 21)
Churn rate: 26.5%

Column data types:
customer_id              str
gender                   str
senior_citizen          bool
has_partner             bool
has_dependents          bool
tenure_months          int64
contract_type            str
paperless_billing       bool
payment_method           str
phone_service           bool
multiple_lines           str
internet_service         str
online_security          str
online_backup            str
device_protection        str
tech_support             str
streaming_tv             str
streaming_movies         str
monthly_charges      float64
total_charges        float64
churned                 bool
dtype: object


In [4]:
# Verify the data matches our SQL findings
print("Churn rate by contract type (sanity check vs SQL):")
print(df.groupby('contract_type')['churned'].agg(['count', 'sum', 'mean']))

Churn rate by contract type (sanity check vs SQL):
                count   sum      mean
contract_type                        
Month-to-month   3875  1655  0.427097
One year         1473   166  0.112695
Two year         1695    48  0.028319
